# Multi-task: an auxiliary head alongside the ramp target

The ramp target is the one thing that paid (+0.0074 cloud). Refining it is exhausted — the
W-sweep and the geometry sweep both say it is already at a plateau optimum. This is the last
structurally different idea in the family that worked: keep the ramp as the primary target but
**train the same trees to predict a second, related quantity at the same time**.

Why it might help: the ramp saturates at 1 once `since > 2W` — after that it carries no
information, so every far-post-break row looks identical to the model. An auxiliary target that
keeps varying (time since break, break magnitude) gives the shared trees a reason to keep
distinguishing those rows, which may sharpen the representation the ramp head reads from.

Mechanism: CatBoost `MultiRMSE` — one model, shared trees, 2-column target. We score with
**column 0 (the ramp)** exactly as the shipped model does; the aux column is discarded at
inference, so **inference cost is unchanged**.

Auxiliary targets (each rescaled to ~[0,1] so it cannot dominate the joint loss):
- **since** — `log1p(steps since break)`, normalised. Keeps growing where the ramp saturates.
- **isbreak** — will this series EVER break (constant per series). A coarser, global task.
- **mag** — the break magnitude (constant per series), from the cached estimate.

Same folds, same 50 features. Gate unchanged: **t >= 3**, then 10 folds before it ships.

Note the aux targets use `tau` at TRAIN time only — same status as the ramp itself. Validation
rows are scored against the hard label `y`, and nothing aux-derived is available at inference.

In [1]:
import os, sys, json, time
import numpy as np, pandas as pd
from catboost import CatBoostRegressor

TOOLS = os.path.abspath('../tools'); sys.path.insert(0, TOOLS)
import cv_tools as CV
REPO = os.path.abspath('../../..')

cache = np.load(os.path.join(TOOLS, 'cv_folds_by_series', 'cv_cache_full.npz'))
X, y, sid, tim = cache['X'], cache['y'], cache['sid'], cache['time']
sampled = cache['is_sampled_step']
index = pd.MultiIndex.from_arrays([sid, tim], names=['id', 'time'])
step = CV.steps_from_index(index)
uids = np.unique(sid)

idx = pd.read_parquet(os.path.join(REPO, 'y_train_index.parquet'))
tau_of = idx['tau_index'].reindex(uids)
tau_row = tau_of.reindex(sid).to_numpy()
since = step - tau_row
is_break = tau_row >= 0

# primary target: the shipped ramp (linear, W=40)
W = 40.0
ramp = np.clip(since / (2.0 * W), 0.0, 1.0)
ramp[~is_break] = 0.0; ramp[is_break & (since < 0)] = 0.0

# --- auxiliary targets, each scaled to ~[0,1] so neither dominates MultiRMSE ---
pos = np.where(is_break & (since >= 0), np.maximum(since, 0), 0)
aux_since = np.log1p(pos); aux_since = aux_since / max(aux_since.max(), 1e-9)

aux_isbreak = is_break.astype(np.float64)

mg = np.load('break_magnitude.npz')          # built by label_geometry.ipynb
mag_of = pd.Series(mg['mag'], index=mg['uid'])
mrow = np.nan_to_num(mag_of.reindex(uids).reindex(sid).to_numpy(), nan=0.0)
aux_mag = np.clip(mrow / max(np.nanquantile(mrow, 0.99), 1e-9), 0.0, 1.0)

AUX = {'since': aux_since, 'isbreak': aux_isbreak, 'mag': aux_mag}
for n, a in AUX.items():
    print(f'aux_{n:8s} range [{a.min():.3f},{a.max():.3f}] mean {a[sampled].mean():.3f} '
          f'| corr with ramp {np.corrcoef(a[sampled], ramp[sampled])[0,1]:+.3f}')

aux_since    range [0.000,1.000] mean 0.065 | corr with ramp +0.969
aux_isbreak  range [0.000,1.000] mean 0.497 | corr with ramp +0.300
aux_mag      range [0.000,1.000] mean 0.093 | corr with ramp +0.125


In [2]:
P = dict(iterations=1500, learning_rate=0.02, depth=6, l2_leaf_reg=3.0,
         bootstrap_type='Bernoulli', subsample=0.8, random_seed=0,
         verbose=0, thread_count=-1)

ramp_s = pd.Series(ramp, index=index)

def fp_single(train, val):
    m = CatBoostRegressor(loss_function='RMSE', **P)
    m.fit(train.X, ramp_s.loc[train.index].to_numpy(), sample_weight=train.w)
    return m.predict(val.X)

def fp_multi(aux_vec):
    aux_s = pd.Series(aux_vec, index=index)
    def _f(train, val):
        Y = np.column_stack([ramp_s.loc[train.index].to_numpy(),
                             aux_s.loc[train.index].to_numpy()])
        m = CatBoostRegressor(loss_function='MultiRMSE', **P)
        m.fit(train.X, Y, sample_weight=train.w)
        # column 0 = the ramp head; the aux head is discarded (inference cost unchanged)
        return np.asarray(m.predict(val.X))[:, 0]
    return _f

folds = CV.series_folds(sid, n_splits=5, seed=0)
OUT = 'multitask_results.json'
done = json.load(open(OUT)) if os.path.exists(OUT) else {}
res = {}
configs = [('ramp_only', fp_single)] + [(f'mt_{n}', fp_multi(a)) for n, a in AUX.items()]
for name, f in configs:
    t0 = time.time(); print(f'\n=== {name} ===', flush=True)
    r = CV.run_cv(X, y, sid, index, f, folds=folds,
                  train_row_mask=sampled, row_step=step, verbose=False)
    res[name] = r
    done[name] = r.fold_scores.tolist(); json.dump(done, open(OUT, 'w'), indent=2)
    print(f'{r.summary(name)} | {time.time()-t0:.0f}s', flush=True)


=== ramp_only ===


ramp_only          mean 0.6089 +/- 0.0157 (sem 0.0070) | OOF 0.6079 | folds: 0.5940  0.6044  0.5958  0.6206  0.6297 | 148s



=== mt_since ===


mt_since           mean 0.6073 +/- 0.0155 (sem 0.0069) | OOF 0.6064 | folds: 0.5946  0.6002  0.5949  0.6175  0.6295 | 254s



=== mt_isbreak ===


mt_isbreak         mean 0.5953 +/- 0.0137 (sem 0.0061) | OOF 0.5940 | folds: 0.5825  0.5925  0.5862  0.5981  0.6173 | 279s



=== mt_mag ===


mt_mag             mean 0.6048 +/- 0.0135 (sem 0.0060) | OOF 0.6035 | folds: 0.5910  0.5997  0.5962  0.6128  0.6241 | 268s


In [3]:
print(CV.summarize(res))
print('\n--- each auxiliary head vs ramp-only ---')
for n in res:
    if n != 'ramp_only':
        print(CV.paired_compare(res[n], res['ramp_only'], n, 'ramp_only'))

ramp_only          mean 0.6089 +/- 0.0157 (sem 0.0070) | OOF 0.6079 | folds: 0.5940  0.6044  0.5958  0.6206  0.6297
mt_since           mean 0.6073 +/- 0.0155 (sem 0.0069) | OOF 0.6064 | folds: 0.5946  0.6002  0.5949  0.6175  0.6295
mt_mag             mean 0.6048 +/- 0.0135 (sem 0.0060) | OOF 0.6035 | folds: 0.5910  0.5997  0.5962  0.6128  0.6241
mt_isbreak         mean 0.5953 +/- 0.0137 (sem 0.0061) | OOF 0.5940 | folds: 0.5825  0.5925  0.5862  0.5981  0.6173

--- each auxiliary head vs ramp-only ---
mt_since - ramp_only: -0.0016 +/- 0.0020 (sem 0.0009, t -1.71, wins 1/5) -> within noise
  per-fold deltas: +0.0006  -0.0042  -0.0009  -0.0032  -0.0001
mt_isbreak - ramp_only: -0.0136 +/- 0.0051 (sem 0.0023, t -5.95, wins 0/5) -> consistent
  per-fold deltas: -0.0115  -0.0120  -0.0096  -0.0226  -0.0124
mt_mag - ramp_only: -0.0041 +/- 0.0031 (sem 0.0014, t -3.00, wins 1/5) -> likely
  per-fold deltas: -0.0030  -0.0047  +0.0004  -0.0078  -0.0056
